# Historical drifter destination prediction

Use an existing **Python 3.12** notebook environment in VS Code and open this notebook from the complete project folder. Run the configuration cell below to install the locked packages and initialize the experiment controls.

Then run the **Complete workflow** cell for the experiments covered by the final report, or go directly to a named cell under **Data preprocessing only**, **Stage 1 only**, or **Stage 2 only**. Each model cell is ready to run; no task name needs to be entered. All existing data preparation and export entries remain available.

**Run All starts the complete report workflow.** Once that workflow or a whole stage has completed successfully, its individual model cells show the existing output instead of repeating the fit in the same kernel. To repeat a completed stage's model, restart the kernel, rerun the configuration cell, and run that model's cell.

Each task prints its status and output folder under `results/reproduced/`. Our team's research data, original results, and frozen model weights are preserved in [data/](../data/), [results/reference/](../results/reference/), and [saved_models/](../saved_models/), respectively. The 517 sealed final-test trajectories are excluded from these experiments. The [README](../README.md) explains the study and delivery status.


## Environment and configuration

Run this cell first. It installs missing or mismatched packages into the **Python running this notebook**, using `pyproject.toml` and `uv.lock`, checks the libraries, and loads the experiment controls. It does not install Python, switch kernels, or run experiments. Internet access is needed when packages must be downloaded.

The original code supports **Apple Silicon, macOS 14 or later**, with Apple's Command Line Tools for the native calculations. Python **3.12** is required; **3.12.14** is the reference patch version. If installation updates a package already loaded in memory, the cell will ask for one restart of the current kernel and a rerun of this cell.

In [ ]:
from pathlib import Path
import importlib.util

project_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "code/environment/configure.py").is_file()
                     and (p / "pyproject.toml").is_file()), None)
if project_root is None:
    raise RuntimeError("Open this notebook from the complete project folder.")

spec = importlib.util.spec_from_file_location(
    "notebook_configuration", project_root / "code/environment/configure.py")
configuration = importlib.util.module_from_spec(spec)
spec.loader.exec_module(configuration)
environment_status = configuration.configure(project_root)

from notebook_entry import NotebookRunner
import workflow

# Report scope: W06 (Stage 1) and W07-W11 (Stage 2).
# These lists affect this notebook only; all original programs remain available.
report_stage1_tasks = (
    "stage1.baselines", "stage1.blend", "stage1.neighbourhood",
    "stage1.drogue", "stage1.markov_inputs", "stage1.markov",
    "stage1.markov_diagnostics", "stage1.velocity",
    "stage1.iteration", "stage1.continuous",
)
report_stage2_tasks = (
    "stage2.step1", "stage2.step2.adaptive", "stage2.step2.allocation",
    "stage2.step2.equal_id", "stage2.step2.partial_id",
    "stage2.step3.spatial", "stage2.step3.adaptive",
    "stage2.step3.quality", "stage2.step3.joint",
    "stage2.step4.seasonal", "stage2.step4.temporal",
    "stage2.step4.seasonal_information", "stage2.step4.direction",
    "stage2.step4.exact_origin", "stage2.step4.position_error",
    "stage2.step4.seasonal_exact", "stage2.step5.gaussian",
    "stage2.step5.gmm", "stage2.step5.mdn12",
)


class ReportNotebookRunner(NotebookRunner):
    """Select report experiments while retaining the original execution programs."""

    def run(self, task, **options):
        suites = {
            "stage1.all": report_stage1_tasks,
            "stage2.all": report_stage2_tasks,
            "full.refit": report_stage1_tasks + report_stage2_tasks,
        }
        if task not in suites:
            return super().run(task, **options)

        followups = ("report.rebuild", "presentation.export") if task == "full.refit" else ()
        required = list(suites[task]) + list(followups)
        if options.get("with_raw_replay"):
            required.insert(0, "data.raw_cleaning")
        previous = [self._completed_result(item) for item in required]
        if all(previous):
            print("These report tasks are already complete in this kernel.")
            return {"task": task, "status": "complete",
                    "reused_output_directories": list(dict.fromkeys(
                        item["output_directory"] for item in previous))}

        print("Running report experiments:", task, flush=True)
        result = workflow.run_task(
            "full.refit", root=self.root, tasks=list(suites[task]),
            task_id=task, **options)
        if result.get("status") != "complete":
            return result
        # Cache completed tasks so Run All does not fit them again below.
        cached = {**result, "completed_tasks": list(result["completed_tasks"])}
        self.completed_suites[task] = cached
        followup_outputs = []
        for item in followups:
            child = super().run(item)
            if child.get("status") != "complete":
                raise RuntimeError("Report output did not complete: " + item)
            cached["completed_tasks"].append({
                "task": item, "status": "complete",
                "receipt": str(Path(child["output_directory"]) / "run.json"),
            })
            followup_outputs.append(child["output_directory"])
        print("Completed. Experiment outputs:", result["output_directory"])
        return {**result, "notebook_followup_outputs": followup_outputs}


notebook = ReportNotebookRunner(project_root, environment_ready=environment_status["ok"])
print("Ready. Run the complete report workflow or any named research cell below.")


## Complete workflow

**Run the next cell to process the development data and run the report's Stage 1 and Stage 2 experiments in order.** It then regenerates the report figures and presentation data. Individual model cells are available below for separate runs.

Cleaning replays the original rules and approved repairs. Each model uses its original fixed input snapshot; newly cleaned files do not replace those inputs automatically. Figures use the published predictions and final frozen model. The notebook runs the models and comparisons described in the final report.

Full fitting can take many hours. The last Gaussian timing check exceeded its original computing-time limit, so that fit currently stops at its budget check. The complete workflow has not yet passed an end-to-end run; see the [reproduction audit](../docs/reproduction_audit.md).


In [2]:
# Run all: data preprocessing → Stage 1 → Stage 2 → report figures and presentation data.
# All experiments retain their original fixed inputs and settings.
full_result = notebook.run("full.refit", with_raw_replay=True)


Running: Refit all completed experiments under their frozen input contracts


KeyboardInterrupt: 

## Data preprocessing only

Run a cell below to inspect or prepare data without starting either modelling stage. These cells use the delivered source data and approved cleaning rules.

### Inspect the delivered data

Check the source identities, development split, and data manifests.

In [3]:
# Run: Inspect the delivered data.
result = notebook.run('data.inspect')


Running: Inspect the delivered data identity and development split
Status: complete
Output: /Users/moonedgethug/Documents/UNSW/DATA3001/work_place/results/reproduced/delivery_20261121T150859_451026Z


### Replay development-data cleaning

Reconstruct development trajectories from the fixed raw source, apply the approved repairs, and compare them with the delivered cleaned data. Outputs are saved in a new run folder.

In [4]:
# Run: Replay development-data cleaning.
result = notebook.run('data.raw_cleaning')


Running: Replay native track cleaning into a new output
Status: complete
Output: /Users/moonedgethug/Documents/UNSW/DATA3001/work_place/results/reproduced/delivery_20261121T150903_834502Z


### Prepare Markov input counts

Rebuild the qualified Markov input counts from the delivered cleaned data. The same preparation is included in the complete Stage 1 run.

In [ ]:
# Run: Prepare Markov input counts.
result = notebook.run('stage1.markov_inputs')


## Stage 1 only

The first research round covers the empirical baselines, regularization, spatial sharing, drogue comparisons, Markov iteration, and hourly continuous models in **W06 / Report Section 4**. The comparison programs retain their original fitting, score calculations, and diagnostics.

Run the whole-stage cell, or an individual model cell below. Each experiment prepares or loads the upstream inputs required by its original contract. The iteration-order comparison includes both grid-based and exact-position propagation, as described in W06.


### Run all Stage 1 report experiments

Run the first-round experiments covered by the report, including Markov input preparation, score calculations, and lag diagnostics.


In [ ]:
# Run: Run all Stage 1 experiments.
result = notebook.run('stage1.all')


### Fit the empirical baseline models

Fit the persistence baseline (B0), pooled-origin baseline (B1), and origin-cell baseline (B2). These three references share one original fitting program. The five-fold baseline scores in report Table 6 are recomputed by the regularized-model comparison below.


In [ ]:
# Run: Fit the empirical baseline models.
result = notebook.run('stage1.baselines')


### Fit and score the baselines and regularized models

Recompute the five-fold B0/B1/B2 comparison in report Table 6, then compare B1/B2 blending using the original nested cross-validation procedure.


In [ ]:
# Run: Fit regularized baseline models.
result = notebook.run('stage1.blend')


### Fit spatial neighbourhood models

Compare borrowing endpoint and displacement information from nearby cells.

In [ ]:
# Run: Fit spatial neighbourhood models.
result = notebook.run('stage1.neighbourhood')


### Compare drogue states

Run the original drogue-regime model comparisons.

In [ ]:
# Run: Compare drogue states.
result = notebook.run('stage1.drogue')


### Fit the daily Markov model

Fit and evaluate the daily Markov baseline using the qualified inputs.

In [ ]:
# Run: Fit the daily Markov model.
result = notebook.run('stage1.markov')


### Run Markov lag comparisons and diagnostics

Compare transition step lengths and reproduce the propagation diagnostics described in W01 and W06, including the selected-step-length scores in report Table 7.


In [ ]:
# Run: Run Markov lag diagnostics.
result = notebook.run('stage1.markov_diagnostics')


### Fit velocity-state Markov models

Reproduce the first-round velocity-state model comparison.

In [ ]:
# Run: Fit velocity-state Markov models.
result = notebook.run('stage1.velocity')


### Compare iteration orders and position representations

Run the reported thirty-day comparison with both grid-based and exact-position propagation. Each representation compares nine iteration orders and eleven mixing weights, giving 99 combinations. This includes the position-refinement result discussed in W06.


In [ ]:
# Run: Compare iteration orders.
result = notebook.run('stage1.iteration')


### Fit hourly continuous models

Reproduce the hourly continuous-model experiments from the first research round.

In [ ]:
# Run: Fit hourly continuous models.
result = notebook.run('stage1.continuous')


## Stage 2 only

The second research round follows the five-step framework below. It moves from discrete transition matrices to continuous endpoint distributions, with later experiments focused on seven-day prediction. Run the whole-stage cell, or any individual model cell.

### Run all Stage 2 experiments

Run the completed experiments in all five steps. The twelve-count network comparison includes the initial six candidate component counts, so those fits run once.

In [ ]:
# Run: Run all Stage 2 experiments.
result = notebook.run('stage2.all')


### Step 1 — Empirical transition matrix

Rebuild the empirical reference for the second research round.

In [ ]:
# Run the Step 1 empirical transition-matrix experiment.
result = notebook.run("stage2.step1")


### Step 2 — Regularized / Bayesian matrix

Compare B1/B2 blending and the contribution assigned to each training drifter.

#### Adaptive baseline blending

Fit the adaptive B1/B2 blending model.

In [ ]:
# Run: Adaptive baseline blending.
result = notebook.run('stage2.step2.adaptive')


#### Allocation of pooled baseline information

Compare how the pooled B1 information contributes to the local prediction.

In [ ]:
# Run: Allocation of pooled baseline information.
result = notebook.run('stage2.step2.allocation')


#### Equal-drifter regularization

Run the regularization comparison with equal total contribution per drifter.

In [ ]:
# Run: Equal-drifter regularization.
result = notebook.run('stage2.step2.equal_id')


#### Partial drifter weighting

Compare intermediate weighting of each drifter's contribution.

In [ ]:
# Run: Partial drifter weighting.
result = notebook.run('stage2.step2.partial_id')


### Step 3 — Spatially structured matrix

Compare endpoint and displacement information from nearby cells, then refine the local information weights.

#### Spatial endpoint and displacement sharing

Compare the two spatial-sharing methods under the original fixed and adaptive settings.

In [ ]:
# Run: Spatial endpoint and displacement sharing.
result = notebook.run('stage2.step3.spatial')


#### Adaptive local endpoint weights

Fit adaptive weights for the neighbouring endpoint information.

In [ ]:
# Run: Adaptive local endpoint weights.
result = notebook.run('stage2.step3.adaptive')


#### Local endpoint information quality

Run the comparison of local endpoint information quality.

In [ ]:
# Run: Local endpoint information quality.
result = notebook.run('stage2.step3.quality')


#### Joint local endpoint constraints

Run the experiment combining the local endpoint constraints.

In [ ]:
# Run: Joint local endpoint constraints.
result = notebook.run('stage2.step3.joint')


### Step 4 — Time- and environment-dependent transitions

Test seasonal and temporal information, departure direction, exact origin, and position uncertainty.

#### Seasonal transition matrices

Fit and compare seasonal transition matrices with pooled data.

In [ ]:
# Run: Seasonal transition matrices.
result = notebook.run('stage2.step4.seasonal')


#### Temporal-information comparison

Reproduce the recorded temporal-information experiment.

In [ ]:
# Run: Temporal-information comparison.
result = notebook.run('stage2.step4.temporal')


#### Seasonal-information diagnostics

Reproduce the analysis of seasonal information and sample support.

In [ ]:
# Run: Seasonal-information diagnostics.
result = notebook.run('stage2.step4.seasonal_information')


#### Departure-direction model

Run the model comparison using departure direction.

In [ ]:
# Run: Departure-direction model.
result = notebook.run('stage2.step4.direction')


#### Exact-origin model

Use exact departure coordinates in the prediction comparison.

In [ ]:
# Run: Exact-origin model.
result = notebook.run('stage2.step4.exact_origin')


#### Position-uncertainty model

Run the experiment using the recorded longitude and latitude uncertainty.

In [ ]:
# Run: Position-uncertainty model.
result = notebook.run('stage2.step4.position_error')


#### Seasonal exact-origin model

Compare seasonal and pooled information with exact departure coordinates.

In [ ]:
# Run: Seasonal exact-origin model.
result = notebook.run('stage2.step4.seasonal_exact')


### Step 5 — Memory-aware / latent-state models

The implemented extension uses continuous Gaussian and mixture distributions. It does not include an explicit latent-state sequence model.

#### Conditional Gaussian model

Fit the conditional Gaussian model. Its original computing-time check remains active.

In [ ]:
# Run: Conditional Gaussian model.
result = notebook.run('stage2.step5.gaussian')


#### Local Gaussian mixture model

Fit and evaluate the local Gaussian mixture distributions.

In [ ]:
# Run: Local Gaussian mixture model.
result = notebook.run('stage2.step5.gmm')


#### Mixture density networks: initial six candidate counts

Train and compare the initial six candidate component counts. This is six network configurations, not a single six-component network.

In [ ]:
# Run: Mixture density networks: initial six candidate counts.
result = notebook.run('stage2.step5.mdn6')


#### Mixture density networks: twelve candidate counts

Train all twelve candidate component counts, including the initial six and the final recommended 22-component model.

In [ ]:
# Run: Mixture density networks: twelve candidate counts.
result = notebook.run('stage2.step5.mdn12')


## Final model and report

These cells use the frozen model or published results. They can run after environment configuration without refitting either research stage.

### Predict with the final 22-component model

Generate seven-day endpoint samples for an origin at 78°W, 33°N and display them. Change the coordinates in this cell to inspect another origin. This loads the frozen weights without training.

In [ ]:
# Predict and plot seven-day endpoints with the frozen final model.
import numpy as np
import matplotlib.pyplot as plt

origins = [[-78.0, 33.0]]  # longitude, latitude, in degrees
prediction = notebook.run("predict.final_mdn22", origins=origins, samples=1000, seed=20261126)
endpoints = np.load(Path(prediction["output_directory"]) / "prediction/endpoint_samples_lonlat.npy")
fig, ax = plt.subplots(figsize=(8, 5))
for index, origin in enumerate(origins):
    points = endpoints[index]
    ax.scatter(points[:, 0], points[:, 1], s=5, alpha=0.2,
               label="Predicted endpoints: origin " + str(index + 1))
    ax.scatter(*origin, marker="*", s=140, edgecolor="black", zorder=5)
ax.set(xlabel="Longitude (degrees)", ylabel="Latitude (degrees)",
       title="Seven-day endpoint samples: frozen 22-component model")
ax.legend()
ax.grid(alpha=0.2)
plt.show()


### Rebuild report figures

Regenerate the report figures from the saved predictions and regional tables. The README also explains how to recompute regional integration using the frozen model.

In [ ]:
# Run: Rebuild report figures.
result = notebook.run('report.rebuild')


### Rebuild presentation data

Regenerate the published presentation data and compare them with the original exports.

In [ ]:
# Run: Rebuild presentation data.
result = notebook.run('presentation.export')


### Recover the reported final-model comparison

Copy the recorded twelve-candidate network comparison and its source identities into a new output folder for inspection alongside W11 / report Table 11. This does not refit models or recompute scores.


In [ ]:
# Recover the final-model comparison reported in Table 11.
result = notebook.run('results.recover', recover_task='stage2.step5.mdn12')


## Source and evidence navigation

- [Data processing](data_preprocessing/) — source acquisition, cleaning, and pair generation.
- [Stage 1 models](modelling/stage1/) and [Stage 2 models](modelling/stage2/) — original scientific implementations.
- [Report generation](reporting/) — figures and supplementary analyses.
- [Reproduction audit](../docs/reproduction_audit.md) — completed checks and remaining execution limits.
- [Migration map](../docs/migration_manifest.json) — old and new paths and source hashes.

Each task prints its output location. In that folder, `run.json` records its status and execution scope; `input_manifest.json` records the inputs. Original experiment programs retain their definitions and checks inside an isolated run directory.